<a href="https://colab.research.google.com/github/Amal1703/AI_ML_football/blob/main/stage_1_train/test_YOLO_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Turn on GPU acceleration

In [ ]:
# In Colab:
# Click Runtime
# Then click Change runtime type
# Under Hardware accelerator, select GPU T4
# Click Save


# Install dependencies

In [ ]:
!uv pip install roboflow
!uv pip install ultralytics

# Import libraries

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO

from pathlib import Path
import requests
import zipfile
import os
from IPython.display import HTML
from base64 import b64encode
import subprocess
import imageio_ffmpeg
import glob

# Download dataset

In [ ]:
# Download the dataset from Roboflow (yolov11)
# 1. from https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/20
# 2. ON Popular Download Formats, cliqu on yolov1
# 3. clic on Download dataset and copy the python code (code au dessous) to dwonlod data

rf = Roboflow(api_key="waNECdEukH3W2rtXwlET")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(20)
dataset = version.download("yolov11")

# Train YOLO 11 model

In [ ]:
!yolo task=detect mode=train data={dataset.location}/data.yaml model=yolo11n.pt epochs=140 imgsz=640 val=True patience=15

# Test the input data (video) using YOLO model

In [ ]:
# Download data

repo = "Amal1703/AI_ML_football"
files = [
   "input_data.zip",
]

dl_dir = Path("colab")
dl_dir.mkdir(exist_ok=True)

def download(name):
    out = dl_dir / name
    for base in ("https://media.githubusercontent.com/media", "https://raw.githubusercontent.com"):
        url = f"{base}/{repo}/main/{name}" if "media" in base else f"{base}/{repo}/main/{name}"
        try:
            with requests.get(url, stream=True, allow_redirects=True, timeout=60) as r:
                r.raise_for_status()
                with open(out, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            print(f"OK   {name}  ({out.stat().st_size/1e6:.1f} MB) <- {url}")
            return out
        except requests.RequestException as e:
            print(f"Failed {url}: {e}")
    print(f"!! Could not download {name}")
    return None

downloaded = [p for p in (download(n) for n in files) if p]

# Bundle everything into colab.zip
with zipfile.ZipFile("colab.zip", "w", zipfile.ZIP_STORED) as z:  # STORED: files are already compressed
    for p in downloaded:
        z.write(p, arcname=p.name)

print("colab.zip created:", Path("colab.zip").stat().st_size / 1e6, "MB")

In [ ]:
with zipfile.ZipFile('/content/colab/input_data.zip', 'r') as zip_ref:
   zip_ref.extractall('/content/colab')

In [ ]:
# load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

In [ ]:
results =  next(model.predict(
    source="/content/colab/input_data/08fd33_4.mp4",
    save=True,
    project=os.getcwd(),   # root output directory
    name='output_data_after_yolo_prediction',      # sub-folder
    exist_ok=True ,             # do not create "output_data_after_yolo_prediction2", etc.
    stream=True  ,
    verbose=False
))

for r in results:
    pass

In [ ]:
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

# Retrieve the .avi path video and convert it to mp4
avi_path = glob.glob(r.save_dir + "/*.avi")[0]
mp4_path = avi_path.replace(".avi", ".mp4")

# Conversion with ffmpeg
subprocess.run([ffmpeg_exe, "-y", "-i", avi_path, "-vcodec", "libx264", mp4_path])

# Display
mp4 = open(mp4_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f'<video width=500 controls><source src="{data_url}" type="video/mp4"></video>')

As you can see in the video, the same referee or goalkeeper is detected as a player, or vice versa.